In [1]:
import pandas as pd
spam_dataset = "/kaggle/input/datasets/meruvulikith/190k-spam-ham-email-dataset-for-classification/spam_Emails_data.csv"
data = pd.read_csv(spam_dataset)

In [2]:
data.head()

,label,text
0,Spam,viiiiiiagraaaa\nonly for the ones that want to...
1,Ham,got ice thought look az original message ice o...
2,Spam,yo ur wom an ne eds an escapenumber in ch ma n...
3,Spam,start increasing your odds of success & live s...
4,Ham,author jra date escapenumber escapenumber esca...


In [3]:
data['label'][:200].value_counts()

label
Ham     103
Spam     97
Name: count, dtype: int64

In [4]:
data['label'] = data['label'].str.strip().str.lower().map({'spam': 1, 'ham': 0})

In [5]:
import numpy as np
from sklearn.model_selection import train_test_split

X = np.array(data['text'][:200])
y = np.array(data['label'][:200])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42)

In [6]:
from sklearn.base import BaseEstimator, TransformerMixin
from collections import Counter
import re
import nltk
stemmer = nltk.PorterStemmer()

class EmailToWordCounterTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, strip_headers=True, lower_case=True,
                 remove_punctuation=True, replace_urls=True,
                 replace_numbers=True, stemming=True):
        self.strip_headers = strip_headers
        self.lower_case = lower_case
        self.remove_punctuation = remove_punctuation
        self.replace_urls = replace_urls
        self.replace_numbers = replace_numbers
        self.stemming = stemming
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        X_transformed = []
        for email in X:
            text = email
            if self.lower_case:
                text = text.lower()
            if self.replace_numbers:
                text = re.sub(r'\d+(?:\.\d*)?(?:[eE][+-]?\d+)?', 'NUMBER', text)
            if self.remove_punctuation:
                text = re.sub(r'\W+', ' ', text, flags=re.M)
            word_counts = Counter(text.split())
            if self.stemming and stemmer is not None:
                stemmed_word_counts = Counter()
                for word, count in word_counts.items():
                    stemmed_word = stemmer.stem(word)
                    stemmed_word_counts[stemmed_word] += count
                word_counts = stemmed_word_counts
            X_transformed.append(word_counts)
        return np.array(X_transformed)

In [7]:
X_few = X_train[:3]
X_few_wordcounts = EmailToWordCounterTransformer().fit_transform(X_few)
X_few_wordcounts

array([Counter({'escapenumb': 4, 'promot': 2, 'it': 2, 'thi': 1, 'one': 1, 'is': 1, 'be': 1, 'take': 1, 'advantag': 1, 'sym': 1, 'chvccurrent': 1, 'day': 1, 'target': 1, 'price': 1, 'escapenumberact': 1, 'strong': 1, 'buy': 1, 'get': 1, 'on': 1, 'befor': 1, 's': 1, 'too': 1, 'late': 1, 'see': 1, 'the': 1, 'news': 1, 'smilenn': 1, 'call': 1, 'your': 1, 'broker': 1}),
       Counter({'bescapenumb': 366, 'cescapenumb': 348, 'descapenumb': 101, 'eescapenumb': 71, 'aescapenumb': 71, 'fescapenumb': 57, 'cf': 52, 'bf': 52, 'bc': 50, 'ba': 41, 'bd': 37, 'be': 27, 'bb': 26, 'ed': 25, 'ce': 20, 'cc': 19, 'ee': 17, 'de': 15, 'ad': 14, 'ae': 14, 'escapenumb': 13, 'ca': 10, 'db': 9, 'fa': 9, 'fd': 9, 'fc': 9, 'df': 8, 'cd': 6, 'fb': 5, 'eb': 4, 'cb': 3, 'af': 3, 'dc': 3, 'aa': 3, 'ac': 3, 'dd': 3, 'da': 3, 'escapelong': 3, 'net': 2, 'ep': 2, 'escapenumberc': 2, 'deni': 2, 'div': 1, 'em': 1, 'ec': 1, 'font': 1, 'ul': 1, 'cqrtescapenumb': 1, 'hanmail': 1, 'nbsp': 1, 'http': 1, 'www': 1, 'findallresul

In [8]:
from scipy.sparse import csr_matrix

class WordCounterToVectorTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, vocabulary_size=1000):
        self.vocabulary_size = vocabulary_size
    def fit(self, X, y=None):
        total_count = Counter()
        for word_count in X:
            for word, count in word_count.items():
                total_count[word] += min(count, 10)
        most_common = total_count.most_common()[:self.vocabulary_size]
        self.vocabulary_ = {word: index + 1
                            for index, (word, count) in enumerate(most_common)}
        return self
    def transform(self, X, y=None):
        rows = []
        cols = []
        data = []
        for row, word_count in enumerate(X):
            for word, count in word_count.items():
                rows.append(row)
                cols.append(self.vocabulary_.get(word, 0))
                data.append(count)
        return csr_matrix((data, (rows, cols)),
                          shape=(len(X), self.vocabulary_size + 1))

In [9]:
from sklearn.pipeline import Pipeline

preprocess_pipeline = Pipeline([
    ("email_to_wordcount", EmailToWordCounterTransformer()),
    ("wordcount_to_vector", WordCounterToVectorTransformer()),
])

X_train_transformed = preprocess_pipeline.fit_transform(X_train)

In [10]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score

mlp_clf = MLPClassifier(
    hidden_layer_sizes=(44,44,),  
    max_iter=300,
    random_state=42
)

# Defaults to accuracy for classifiers
score = cross_val_score(mlp_clf, X_train_transformed, y_train, cv=3, scoring="accuracy")

print("Accuracy:", score.mean())

Accuracy: 0.8252969951083159


In [11]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, confusion_matrix

y_pred = cross_val_predict(mlp_clf, X_train_transformed, y_train, cv=3)

print("Confusion Matrix:\n", confusion_matrix(y_train, y_pred))
print(f"Precision: {precision_score(y_train, y_pred):.4f}")
print(f"Recall:    {recall_score(y_train, y_pred):.4f}")

Confusion Matrix:
 [[71 14]
 [14 61]]
Precision: 0.8133
Recall:    0.8133


In [12]:
import re
import time
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

t0 = time.time()

data = pd.read_csv(spam_dataset).dropna(subset=['text', 'label'])
data['label'] = data['label'].str.strip().str.lower().map({'spam': 1, 'ham': 0})
data = data.dropna(subset=['label'])

def clean_text(text):
    # Truncate huge hex/log emails first so regex and vectorization stay fast
    text = str(text)[:3000].lower()
    text = re.sub(r'escapenumber|escapelong', ' ', text)
    return text
data['clean_text'] = data['text'].map(clean_text)
data = data.drop_duplicates(subset=['clean_text'])

X = data['text'].to_numpy()
y = data['label'].astype(int).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = Pipeline([
    ('tfidf', TfidfVectorizer(
    preprocessor=clean_text,
    ngram_range=(1, 2),
    max_features=30000,
    sublinear_tf=True,
    min_df=3
)),
    ('clf', LogisticRegression(C=2.0, max_iter=200))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Done in {time.time() - t0:.1f}s")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=4))

Done in 95.1s
Confusion Matrix:
 [[19960   431]
 [  243 17980]]
              precision    recall  f1-score   support

           0     0.9880    0.9789    0.9834     20391
           1     0.9766    0.9867    0.9816     18223

    accuracy                         0.9825     38614
   macro avg     0.9823    0.9828    0.9825     38614
weighted avg     0.9826    0.9825    0.9825     38614



In [13]:
probs = model.predict_proba(X_test)[:, 1]
y_pred_strict = (probs >= 0.75).astype(int)
print(confusion_matrix(y_test, y_pred_strict))

[[20210   181]
 [  807 17416]]


In [14]:
import numpy as np
from sklearn.metrics import precision_score, recall_score

probs = model.predict_proba(X_test)[:, 1]

for t in [0.50, 0.60, 0.65, 0.70, 0.75, 0.80]:
    preds = (probs >= t).astype(int)
    fp = np.sum((y_test == 0) & (preds == 1))
    fn = np.sum((y_test == 1) & (preds == 0))
    p = precision_score(y_test, preds)
    r = recall_score(y_test, preds)
    print(f"t={t:.2f} | FP: {fp:3d} | FN: {fn:3d} | Precision: {p:.4f} | Recall: {r:.4f}")

t=0.50 | FP: 431 | FN: 243 | Precision: 0.9766 | Recall: 0.9867
t=0.60 | FP: 331 | FN: 405 | Precision: 0.9818 | Recall: 0.9778
t=0.65 | FP: 277 | FN: 511 | Precision: 0.9846 | Recall: 0.9720
t=0.70 | FP: 220 | FN: 627 | Precision: 0.9877 | Recall: 0.9656
t=0.75 | FP: 181 | FN: 807 | Precision: 0.9897 | Recall: 0.9557
t=0.80 | FP: 131 | FN: 1049 | Precision: 0.9924 | Recall: 0.9424


In [15]:
import joblib

artifact = {
    "pipeline": model,
    "threshold": 0.70,
}
joblib.dump(artifact, "spam_classifier.joblib")

# Later, during inference:
loaded = joblib.load("spam_classifier.joblib")
sample_emails = [
    "Congratulations! Claim your free prize now by clicking http://promo.link",
    "Hey, can you review the Q3 SQL migration script before tomorrow's standup?"
]
probs = loaded["pipeline"].predict_proba(sample_emails)[:, 1]
preds = (probs >= loaded["threshold"]).astype(int)

for email, prob, pred in zip(sample_emails, probs, preds):
    label = "SPAM" if pred == 1 else "HAM"
    print(f"[{label} | p={prob:.3f}] {email[:60]}...")

[SPAM | p=0.959] Congratulations! Claim your free prize now by clicking http:...
[HAM | p=0.148] Hey, can you review the Q3 SQL migration script before tomor...


In [16]:
vec = model.named_steps['tfidf']
clf = model.named_steps['clf']
feature_names = vec.get_feature_names_out()

sample = clean_text("Congratulations! Claim your free prize now by clicking http://promo.link")
x_vec = vec.transform([sample])
for idx in x_vec.indices:
    print(f"{feature_names[idx]:20s} | tfidf={x_vec[0, idx]:.3f} | weight={clf.coef_[0, idx]:+.3f}")

by                   | tfidf=0.129 | weight=+0.269
by clicking          | tfidf=0.336 | weight=-0.565
claim                | tfidf=0.262 | weight=+1.783
claim your           | tfidf=0.396 | weight=+0.405
clicking             | tfidf=0.306 | weight=-0.772
congratulations      | tfidf=0.302 | weight=-3.170
free                 | tfidf=0.173 | weight=+0.836
http                 | tfidf=0.108 | weight=+8.441
link                 | tfidf=0.214 | weight=+1.096
now                  | tfidf=0.161 | weight=+3.710
prize                | tfidf=0.305 | weight=+1.171
promo                | tfidf=0.363 | weight=+0.450
your                 | tfidf=0.117 | weight=+5.470
your free            | tfidf=0.330 | weight=-1.945
